# Day 4 | Lab 4.1: Document Loaders & Chunking — The RAG Front-End

**Duration:** ~1.5 hours

**Scenario.** Mixed-document ingestion for a financial-research workspace — PDFs of analyst reports, CSVs of transactions, JSON conversation logs, structured Markdown policy docs (text, Markdown, CSV, JSON, PDFs of arXiv papers, Word docs).

**Learning Objectives.** By the end of this lab, you will be able to:
1. Pick the right `DocumentLoader` for a given source type (text, Markdown, CSV, JSON, PDFs, directories).
2. Compare PDF loaders side-by-side: `PyPDFLoader` (fast, page-level) vs `PyMuPDFLoader` (richer metadata) vs `UnstructuredPDFLoader` (layout-aware, table detection).
3. Use `RecursiveCharacterTextSplitter` correctly — pick `chunk_size`, `chunk_overlap`, and separators.
4. Apply structural splitters (`MarkdownHeaderTextSplitter`, code splitters) when the document has known structure.
5. Use **tiktoken-based** splitters to enforce a hard token budget per chunk (e.g., embedding model max).
6. Decide between fixed-size, recursive, structural, and semantic chunking — what trade-off you're making.

**Tools.** LangChain v1 · `langchain-community` (loaders) · `langchain-text-splitters` · `tiktoken` · `unstructured`.

Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


## 0. Install Dependencies & System Requirements
Run this cell first to ensure all loaders, parsers, and OCR engines are available in the Colab environment.

In [ ]:
# !pip install -qU python-dotenv langchain langchain-community langchain-classic langchain-openai langchain-text-splitters "unstructured[all-docs]" python-docx pypdf pymupdf pandas jq tiktoken faiss-cpu rank_bm25 gdown unstructured-pytesseract

In [ ]:
# Install system-level dependencies required for Unstructured to process PDFs and perform OCR
# !apt-get update -qq
# !apt-get install -y -qq tesseract-ocr poppler-utils

## 1. API Key Configuration

In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv("..\\.env")
except ImportError:
    pass

for key in ['OPENAI_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


---
## 2. Document Loaders — One Example Per Source Type

Every RAG pipeline starts with parsing source data into LangChain `Document` objects. Each loader returns `Document(page_content=..., metadata={...})`. The metadata dict is what later powers citations, filters, and source attribution.

We'll cover one canonical example per source type: text, Markdown, CSV, JSON, three PDF loaders (the choice matters), Word documents, and directory loaders for batch ingestion.


### 2.1 Text Loader

The simplest loader — one file → one `Document`.

In [ ]:
!curl -L -o README.md https://raw.githubusercontent.com/langchain-ai/langchain/master/README.md

In [ ]:
from langchain_community.document_loaders import TextLoader

# Explicitly declaring utf-8 encoding prevents UnicodeDecodeError on Windows/Mojibake
loader = TextLoader("./README.md", encoding="utf-8")
doc = loader.load()

In [ ]:
print(doc[0].page_content[:1000])

### 2.2 Markdown Loader (`UnstructuredMarkdownLoader`)

For `.md` files. By default returns a single document; pass `mode='elements'` to split by element (heading, paragraph, list, table). We'll show the single-document mode here; for header-aware splitting we use `MarkdownHeaderTextSplitter` later (Section 6).

In [ ]:
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader

# Explicitly declaring utf-8 encoding ensures special markdown characters render correctly
loader = UnstructuredMarkdownLoader("./README.md", mode='single', encoding="utf-8")
docs = loader.load()

In [ ]:
print(docs[0].page_content[:1000])

### 2.3 CSV Loader

`CSVLoader` returns one `Document` per row; the row's columns become the page content and the source metadata. Use `csv_args` to customize the delimiter / quotechar.

### CSV Loader

A comma-separated values (CSV) file is a delimited text file that uses a comma to separate values. Each line of the file is a data record. Each record consists of one or more fields, separated by commas.

LangChain implements a CSV Loader that will load CSV files into a sequence of `Document` objects. Each row of the CSV file is converted to one document.

In [ ]:
import pandas as pd

# Create a DataFrame with some dummy real estate data
data = {
    'Property_ID': [101, 102, 103, 104, 105],
    'Address': ['123 Elm St', '456 Oak St', '789 Pine St', '321 Maple St', '654 Cedar St'],
    'City': ['Springfield', 'Rivertown', 'Laketown', 'Hillside', 'Sunnyvale'],
    'State': ['CA', 'TX', 'FL', 'NY', 'CO'],
    'Zip_Code': [98765, 87654, 76543, 65432, 54321],
    'Bedrooms': [3, 2, 4, 3, 5],
    'Bathrooms': [2, 1, 3, 2, 4],
    'Listing_Price': [500000, 350000, 600000, 475000, 750000]
}

df = pd.DataFrame(data)

# Save the DataFrame to a CSV file
df.to_csv('data.csv', index=False)

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path="./data.csv")
docs = loader.load()

In [ ]:
print(docs[0].page_content)

`CSVLoader` will accept a `csv_args` kwarg that supports customization of arguments passed to Python's csv.`DictReader`. See the [`csv` module](https://docs.python.org/3/library/csv.html) documentation for more information of what `csv` args are supported.

In [ ]:
loader = CSVLoader(file_path="./data.csv",
                   csv_args={
                      "delimiter": ",",
                      "quotechar": '"',
                      "fieldnames": ["Property ID", "Address", "City", "State",
                                     "Zip Code", "Bedrooms", "Bathrooms", "Price"],
                   },
                  )
docs = loader.load()

### 2.4 JSON Loader

`JSONLoader` extracts content via a [jq](https://jqlang.org/) selector — `jq_schema='.messages[].content'` pulls the `content` field of every object inside `messages`.

### JSON Loader

[JSON (JavaScript Object Notation)](https://en.wikipedia.org/wiki/JSON) is an open standard file format and data interchange format that uses human-readable text to store and transmit data objects consisting of attribute–value pairs and arrays (or other serializable values).

[JSON Lines](https://jsonlines.org/) is a file format where each line is a valid JSON value.

LangChain implements a [JSONLoader](https://api.python.langchain.com/en/latest/document_loaders/langchain_community.document_loaders.json_loader.JSONLoader.html) to convert JSON and JSONL data into LangChain `Document` objects. It uses a specified [`jq` schema](https://en.wikipedia.org/wiki/Jq_(programming_language)) to parse the JSON files, allowing for the extraction of specific fields into the content and metadata of the LangChain Document.

It uses the `jq` python package. Check out [this manual](https://jqlang.github.io/jq/manual/) for a detailed documentation of the `jq` syntax.

In [ ]:
import json

# Sample data dictionary similar to the one you provided but with modified contents
data = {
    'image': {'creation_timestamp': 1675549016, 'uri': 'image_of_the_meeting.jpg'},
    'is_still_participant': True,
    'joinable_mode': {'link': '', 'mode': 1},
    'magic_words': [],
    'messages': [
        {'content': 'See you soon!',
         'sender_name': 'User B',
         'timestamp_ms': 1675597571851},
        {'content': 'Thanks for the update! See you then.',
         'sender_name': 'User A',
         'timestamp_ms': 1675597435669},
        {'content': 'Actually, the green one is sold out.',
         'sender_name': 'User B',
         'timestamp_ms': 1675596277579},
        {'content': 'I was hoping to purchase the green one!',
         'sender_name': 'User A',
         'timestamp_ms': 1675595140251},
        {'content': 'I’m really interested in the green one, not the red!',
         'sender_name': 'User A',
         'timestamp_ms': 1675595109305},
        {'content': 'Here’s the $150 for it.',
         'sender_name': 'User B',
         'timestamp_ms': 1675595068468},
        {'photos': [{'creation_timestamp': 1675595059,
                     'uri': 'image_of_the_item.jpg'}],
         'sender_name': 'User B',
         'timestamp_ms': 1675595060730},
        {'content': 'It typically sells for at least $200 online',
         'sender_name': 'User B',
         'timestamp_ms': 1675595045152},
        {'content': 'How much are you asking?',
         'sender_name': 'User A',
         'timestamp_ms': 1675594799696},
        {'content': 'Good morning! $50 is far too low.',
         'sender_name': 'User B',
         'timestamp_ms': 1675577876645},
        {'content': 'Hello! I’m interested in the item you posted. I can offer $50. Let me know if that works for you. Thanks!',
         'sender_name': 'User A',
         'timestamp_ms': 1675549022673}
    ],
    'participants': [{'name': 'User A'}, {'name': 'User B'}],
    'thread_path': 'inbox/User A and User B chat',
    'title': 'User A and User B chat'
}

# Save the modified data to a JSON file
with open('chat_data.json', 'w') as file:
    json.dump(data, file, indent=4)


To load the full data as a single document

In [ ]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path='./chat_data.json',
    jq_schema='.',
    text_content=False)

data = loader.load()

Suppose we are interested in extracting the values under the `messages` key of the JSON data

In [ ]:
loader = JSONLoader(
    file_path='./chat_data.json',
    jq_schema='.messages[]',
    text_content=False)

data = loader.load()
data

Suppose we are interested in extracting the values under the `content` field within the `messages` key of the JSON data

In [ ]:
loader = JSONLoader(
    file_path='./chat_data.json',
    jq_schema='.messages[].content',
    text_content=False)

data = loader.load()
data

### 2.5 PDF Loaders — Three Choices, Three Trade-offs

PDFs are the workhorse format for analyst reports, policy docs, and research papers. There are three loaders worth knowing — they differ on **speed**, **metadata richness**, and **layout awareness**.

| Loader | Speed | Metadata | Layout | Use when |
|---|---|---|---|---|
| `PyPDFLoader` | ⚡ Fast | Page number | Text only | You need quick page-level extraction; clean text-only PDFs |
| `PyMuPDFLoader` | ⚡⚡ Fastest | Rich (page, file metadata, format) | Text only | You need maximum throughput + metadata; can't handle complex layouts |
| `UnstructuredPDFLoader` | 🐢 Slow | Element category (Title, NarrativeText, Table) | Layout-aware | You need tables, sections, headers detected as separate elements |


#### `PyPDFLoader`

In [ ]:
!curl -L -o layoutparser_paper.pdf https://arxiv.org/pdf/2103.15348.pdf

#### PyPDFLoader

Here we load a PDF using `pypdf` into list of documents, where each document contains the page content and metadata with page number. Typically each PDF page becomes one document

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./layoutparser_paper.pdf")
pages = loader.load()

In [ ]:
print(pages[0].page_content)

#### `PyMuPDFLoader`

Note the richer `metadata` — file format, creation date, modification date — vs PyPDF's page-only metadata.

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("./layoutparser_paper.pdf")
pages = loader.load()

In [ ]:
print(pages[0].page_content)

#### `UnstructuredPDFLoader`

Layout-aware. Each element is tagged with a `category` (Title, NarrativeText, Table, ListItem, Footer). Critical for structured RAG — you can keep just `Table` elements for table QA, or filter out `Footer` to remove pagination noise.

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader

loader = UnstructuredPDFLoader('./layoutparser_paper.pdf')
data = loader.load()

In [ ]:
print(data[0].page_content[:1000])

With `strategy='hi_res'` and `mode='elements'`, you get one document per layout element.

In [ ]:
# takes 3-4 mins on Colab
loader = UnstructuredPDFLoader('./layoutparser_paper.pdf',
                               strategy='hi_res',
                               extract_images_in_pdf=False,
                               infer_table_structure=True,
                               chunking_strategy="by_title",
                               max_characters=4000, # max size of chunks
                               new_after_n_chars=3800, # preferred size of chunks
                               combine_text_under_n_chars=2000, # smaller chunks < 2000 chars will be combined into a larger chunk
                               mode='elements')
data = loader.load()

In [ ]:
[doc.metadata['category'] for doc in data]

### 2.6 Directory Loader

`DirectoryLoader` walks a folder and applies a chosen loader to each matching file. Pass `loader_cls=...` to pick the loader; `glob='**/*.pdf'` to filter by extension.

In [ ]:
!curl -L -o "Vision Transformers.pdf" "https://arxiv.org/pdf/2010.11929.pdf"

In [ ]:
from langchain_community.document_loaders import UnstructuredWordDocumentLoader

# Define a dictionary to map file extensions to their respective loaders
loaders = {
    '.pdf': (PyMuPDFLoader, {}),
    '.docx': (UnstructuredWordDocumentLoader, {'strategy': 'fast',
                                              'chunking_strategy' : 'by_title',
                                              'max_characters' : 3000, # max limit of a document chunk
                                              'new_after_n_chars' : 2500, # preferred document chunk size
                                              'mode' : 'elements'
                                              })
}

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

# Define a function to create a DirectoryLoader for a specific file type
def create_directory_loader(file_type, directory_path):
    return DirectoryLoader(
        path=directory_path,
        glob=f"**/*{file_type}",
        loader_cls=loaders[file_type][0],
        loader_kwargs=loaders[file_type][1],
        show_progress=True
    )

# Create DirectoryLoader instances for each file type
pdf_loader = create_directory_loader('.pdf', './')
docx_loader = create_directory_loader('.docx', './')

# Load the files
pdf_documents = pdf_loader.load()
docx_documents = docx_loader.load()

---
## 3. Loader Selection — When to Use Which

| Source type | Loader | One-line reason |
|---|---|---|
| Plain text (`.txt`) | `TextLoader` | Smallest, no parsing overhead |
| Markdown (`.md`) | `UnstructuredMarkdownLoader` (single) + `MarkdownHeaderTextSplitter` (split) | Two-stage: load whole, split by headers |
| CSV (`.csv`) | `CSVLoader` | One Document per row preserves row identity |
| JSON (`.json`) | `JSONLoader(jq_schema=...)` | Cleanest selector for nested fields |
| PDF (text-only, fast) | `PyMuPDFLoader` | Best speed/metadata trade |
| PDF (layout / tables) | `UnstructuredPDFLoader(mode='elements', strategy='hi_res')` | Element categories enable structured RAG |
| Word (`.docx`) | `UnstructuredWordDocumentLoader(mode='elements')` | Sections preserved |
| Folder of mixed types | `DirectoryLoader(loader_cls=...)` | One loader pass per type |
| HTML (web pages) | `WebBaseLoader` / `AsyncHtmlLoader` + `BeautifulSoupTransformer` | Fetch + clean in one pipeline |
| Code repos | `GitLoader` + language-specific code splitter | Preserves function/class boundaries |


---
## 4. Document Splitters & Chunkers — The Why

**Why split at all?** Three reasons:
1. **Embedding model context limits.** Most embedding models cap at ~8K tokens; many at 512.
2. **Retrieval relevance.** A 100-page document averaged into one embedding is useless. Smaller chunks → more focused embeddings → better retrieval.
3. **LLM context budget.** You retrieve top-k chunks and stuff them into the prompt; small chunks let you fit more diverse evidence.

**The trade-off you're tuning:**
- *Too small* (< 200 tokens) → fragments lose their context, retrieval gets noisy.
- *Too large* (> 1500 tokens) → embeddings blur, prompt fills up fast, only top-1 useful.
- **Sweet spot for most RAG: 500–1000 tokens, with 10–15% overlap.** Tune empirically against an eval set.


### 4.1 `RecursiveCharacterTextSplitter` — The Default

`RecursiveCharacterTextSplitter` is the default for prose. It tries to split on a **list of separators** in order: `['\n\n', '\n', ' ', '']`. It splits on the first separator that produces chunks ≤ `chunk_size`. This preserves paragraph and sentence boundaries when possible — that's why it usually wins.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc = """Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.
On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.
Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusiasm. Locals prepare months in advance, cultivating crops and crafting goods for the occasion. The festival features a parade, various competitions, and a night market that lights up the town with vibrant colors and joyous energy.
"""



In [ ]:
print(doc)

Smaller `chunk_size` → more, smaller chunks. Larger → fewer, bigger chunks.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    separators= ["\n\n", "\n", " ", ""],
    chunk_size=300,
    chunk_overlap=0,
)

In [ ]:
texts = text_splitter.split_text(doc)
print(len(texts)) # 5

`chunk_overlap` mitigates information loss when context is divided between chunks (critical for retrieval — a sentence that straddles the boundary survives in both).

`chunk_overlap` helps to mitigate loss of information when context is divided between chunks especially for really small chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    separators= ["\n\n", "\n", " ", ""],
    chunk_size=300,
    chunk_overlap=100,
)

texts = text_splitter.split_text(doc)
print(len(texts)) # 5

In [ ]:
for text in texts:
    print(text)
    print(len(text))
    print()

`create_documents` returns LangChain `Document` objects directly (with metadata) instead of plain strings.

You can create LangChain `Document` chunks with the `create_documents` function

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    separators= ["\n\n", "\n", " ", ""],
    chunk_size=500,
    chunk_overlap=100,
)

In [ ]:
docs = text_splitter.create_documents([doc])
docs

### 4.2 Code Splitters — Language-Aware Splitting

For source code, pass `Language.PYTHON` (or other) to split on syntactically meaningful boundaries — function/class definitions, not arbitrary character counts.

### Code Splitters

`RecursiveCharacterTextSplitter` includes pre-built lists of separators that are useful for splitting text in a specific programming language.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

python_code = """
def hello_world():
    print("Hello, World!")
hello_world()
"""

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=50, chunk_overlap=0
)
python_docs = python_splitter.create_documents([python_code])
python_docs

### 4.3 Markdown Header Splitter — Structural Awareness

### Markdown Splitters

We might want to chunk a document based on the structure. For example, a markdown file is organized by headers. Creating chunks within specific header groups is an intuitive idea. To address this challenge, we can use MarkdownHeaderTextSplitter. This will split a markdown file by a specified set of headers.

For example, if we want to split this markdown:

```
markdown_document = """
# Team Introductions

## Management Team

Hi, this is Jim, the CEO.  
Hi, this is Joe, the CFO.

## Development Team

Hi, this is Molly, the Lead Developer.
"""
```

We can specify the headers to split on:

```
[("#", "Header 1"),
 ("##", "Header 2")]

```

And content is grouped or split by common headers:

```
Document(page_content='Hi, this is Jim, the CEO.\nHi, this is Joe, the CFO.',
metadata={'Header 1': 'Team Introductions', 'Header 2': 'Management Team'})

Document(page_content='Hi, this is Molly, the Lead Developer.',
metadata={'Header 1': 'Team Introductions', 'Header 2': 'Development Team'})
```

In [ ]:
markdown_document = """
# Team Introductions

## Management Team
Hi, this is Jim, the CEO.
Hi, this is Joe, the CFO.

## Development Team
Hi, this is Molly, the Lead Developer.
"""

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

By default, `MarkdownHeaderTextSplitter` strips headers being split on from the output chunk's content. This can be disabled by setting `strip_headers = False`.

In [ ]:
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

### 4.4 tiktoken-Based Splitters — Hard Token Budget

Character-based splitters give *approximate* token counts. When you need a hard cap (e.g., your embedding model maxes at 512 tokens), use a tiktoken-based splitter — it counts tokens exactly, using the same tokenizer the model uses.

In [ ]:
doc = """Welcome to Green Valley, a small town nestled in the heart of the mountains. With its picturesque landscapes and vibrant community life, Green Valley has been a hidden gem for years. The main street is lined with an array of shops and cafes, each offering a unique taste of local flavor and culture.
On a typical afternoon, the town square comes alive with the bustling sounds of locals and visitors mingling. Children play near the fountain, artists display their crafts, and an old man tells stories of days gone by. The aroma of freshly baked bread wafts from the bakery, drawing a steady stream of customers.
Green Valley is not only known for its scenic beauty but also for its annual festivals. The most anticipated event is the Harvest Festival, celebrated with great enthusiasm. Locals prepare months in advance, cultivating crops and crafting goods for the occasion. The festival features a parade, various competitions, and a night market that lights up the town with vibrant colors and joyous energy.
"""

In [ ]:
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(model_name='gpt-4o-mini',
                                  chunk_size=30,
                                  chunk_overlap=10)

docs = text_splitter.create_documents([doc])

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")
for d in docs:
  print('Words:', len(d.page_content.split(' ')),
        'Tokens:', len(enc.encode(d.page_content)),
        'Chunk:', d.page_content)

Larger chunk size in terms of number of words \ tokens will create lesser chunks or paragraphs as usual

In [ ]:
text_splitter = TokenTextSplitter(model_name='gpt-4o-mini',
                                  chunk_size=100,
                                  chunk_overlap=30)

docs = text_splitter.create_documents([doc])

`from_tiktoken_encoder(...)` enforces an exact-token cap on top of the recursive splitter — get both natural boundaries AND a hard token budget.

To implement a hard constraint on the chunk size, we can use `RecursiveCharacterTextSplitter.from_tiktoken_encoder`, where each split will be recursively split if it has a larger size and it makes the chunks more meaningful

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-4o-mini",
    chunk_size=100,
    chunk_overlap=30,
)

docs = text_splitter.create_documents([doc])

In [ ]:
enc = tiktoken.encoding_for_model("gpt-4o-mini")
for d in docs:
  print('Words:', len(d.page_content.split(' ')),
        'Tokens:', len(enc.encode(d.page_content)),
        'Chunk:', d.page_content)

---
## 5. NEW: Section-Aware Chunking — When Structure Matters

**The problem.** Naive recursive chunking on a structured document (a banking policy with headed sections like `## Eligibility`, `## Documentation Required`, `## Interest Rate`) destroys the section context. A query like *"what's the interest rate for ...?"* may retrieve a chunk that's mid-section with no header — the LLM has no idea which section it came from, so it can't answer reliably.

**The fix.** `MarkdownHeaderTextSplitter` keeps each chunk's header chain in metadata. The retrieved chunk arrives with `metadata={'Section': 'Interest Rate', 'Subsection': 'Floating'}` — that context lets the LLM ground every answer.


In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

policy_doc = '''# Personal Loan Policy v3.2

## Eligibility
Indian residents aged 21–60 with minimum monthly income ₹25,000.
Self-employed applicants must show 3 years of ITRs.

## Interest Rate
### Floating Rate
RPLR-linked, currently 12.5% p.a. for salaried customers.
Add 0.5% premium for self-employed.

### Fixed Rate
13.75% p.a. for full tenure. Available only for tenures ≤ 3 years.

## Documentation Required
PAN card, Aadhaar, latest 3 months salary slips, 6 months bank statement, Form 16 for the previous financial year.

## Prepayment Charges
Nil for floating-rate loans. Fixed-rate: 2% of outstanding principal.'''

# --- Naive recursive splitting — context-blind ---
naive = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
naive_chunks = naive.create_documents([policy_doc])
print(f'Naive split → {len(naive_chunks)} chunks')
for i, c in enumerate(naive_chunks):
    print(f'\n--- chunk {i} ---')
    print(f'metadata: {c.metadata}')
    print(c.page_content[:120])


In [ ]:
# --- Header-aware splitting — preserves section context in metadata ---
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ('#', 'Document'),
        ('##', 'Section'),
        ('###', 'Subsection'),
    ],
    strip_headers=False,  # keep the heading inline so the LLM sees it too
)
structured_chunks = header_splitter.split_text(policy_doc)

print(f'Header-aware split → {len(structured_chunks)} chunks')
for i, c in enumerate(structured_chunks):
    print(f'\n--- chunk {i} ---')
    print(f'metadata: {c.metadata}')
    print(c.page_content[:160])


**What just happened.** Each chunk now carries its `Section` and `Subsection` in metadata. When the retriever returns *"RPLR-linked, currently 12.5% p.a."*, the LLM sees that this came from `Section: Interest Rate · Subsection: Floating Rate` — no ambiguity, no hallucinated context.

**Production pattern.** Two-pass split for long structured docs:
1. `MarkdownHeaderTextSplitter` → preserve headers in metadata.
2. `RecursiveCharacterTextSplitter` over each section → enforce token budget.

Same trick works for HTML headings (`HTMLHeaderTextSplitter`) and code (`Language.PYTHON` splitter).


---
## 6. Splitter Selection — When to Use Which

| Document type | Splitter | Why |
|---|---|---|
| Generic prose, unknown structure | `RecursiveCharacterTextSplitter` | Default; preserves paragraphs |
| Embedding model has hard token limit | `RecursiveCharacterTextSplitter.from_tiktoken_encoder(...)` | Counts tokens exactly |
| Markdown with headers | `MarkdownHeaderTextSplitter` (then recursive over each section) | Preserves section context |
| HTML page | `HTMLHeaderTextSplitter` | Preserves DOM headings |
| Source code | `RecursiveCharacterTextSplitter.from_language(Language.PYTHON, ...)` | Splits on def/class boundaries |
| Long CSV (thousands of rows) | Don't split row content; chunk *across* rows by `chunk_size` of N rows | Each chunk = N rows of context |
| PDF research papers | `UnstructuredPDFLoader(mode='elements')` does both load + element-split | Element categories preserve structure |
| Sub-second user query, semantic chunking | `SemanticChunker` (langchain-experimental) | Embedding-aware boundary detection — slow but topical |

> **Default rule.** Start with `RecursiveCharacterTextSplitter` (`chunk_size=800`, `chunk_overlap=100`). Move to a structural splitter only when retrieval quality fails for documents that *have* exploitable structure.


---
## 7. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **Loaders return Documents** | `Document(page_content, metadata)` — metadata becomes citations, filters, source attribution |
| **PDF loader choice matters** | PyMuPDF for speed + metadata; Unstructured for layout/tables; PyPDF for the simple case |
| **RecursiveCharacterTextSplitter is the default** | Tries `\n\n` → `\n` → ` ` → `` to keep paragraph boundaries when possible |
| **Chunk overlap exists for context preservation** | 10–15% overlap mitigates information loss across chunk boundaries |
| **tiktoken splitters enforce hard token budgets** | Use when embedding model has a strict token cap (e.g., 512 for some models) |
| **Section-aware splitting beats naive on structured docs** | MarkdownHeaderTextSplitter preserves headings as metadata — section context survives retrieval |
| **Code & HTML splitters exist for the same reason** | When the document type has known structural anchors, exploit them |

**Next Lab:** Lab 4.2 — Embeddings & Vector Stores (FAISS · ChromaDB · Titan v2) 🧮


## 8. Stretch Exercise (Optional)

1. Build a hybrid loader that walks a folder of mixed PDFs + Word + Markdown and uses `DirectoryLoader` with `loader_cls` per extension. Save the resulting Documents and report the counts per source type.
2. Take a long banking policy doc (or generate one with an LLM) with 5+ sections. Compare retrieval quality with naive recursive splitting vs `MarkdownHeaderTextSplitter` over a 10-question eval set. Report wins, losses, and ties.
3. Build a `chunk_size` sweep: try 200, 500, 1000, 2000 on the same eval set with the same embedding model. Plot retrieval Recall@5 vs chunk size.
4. Implement a CSV-row-aware chunker: each chunk = N rows × M columns (header repeated each chunk). Test against `CSVLoader` + recursive splitting.
5. Use the `SemanticChunker` from `langchain_experimental` and compare its boundary placement against `RecursiveCharacterTextSplitter` on a long technical doc.
6. Wire the section-aware splitter from §5 to a real RAG pipeline (preview of Lab 4.3) — verify the LLM cites the section header in every answer.


---

## Interview Preparation

The questions below mirror what client interviewers commonly ask about the topics in this lab. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. PyPDFLoader vs PyMuPDFLoader vs UnstructuredPDFLoader — when use which?**

*Hint:* Speed vs metadata vs layout-awareness.

*Answer sketch:* PyPDF: simplest, page-level, text-only — use when PDFs are clean and you just need fast page extraction. PyMuPDF: fastest, richer file metadata (creation/mod dates) — production default for text-only PDFs. Unstructured: slowest but layout-aware — categorises every element (Title, NarrativeText, Table, ListItem). Use when tables / figures / sections matter, or when you want layout-driven chunking.

---

**Q2. Why is RecursiveCharacterTextSplitter usually the default — what does it actually do?**

*Hint:* Look at the `separators` list and the recursion order.

*Answer sketch:* It tries to split on a **prioritised list of separators** — `['\n\n', '\n', ' ', '']`. For each candidate it checks: do the resulting chunks fit `chunk_size`? If yes, take it; if no, recurse into the chunks with the next separator. This preserves paragraph → sentence → word → character boundaries in that order. Result: chunks rarely cut mid-word and usually respect paragraph structure — exactly what you want for prose.

---

**Q3. Fixed-size vs recursive vs semantic chunking — how do you choose?**

*Hint:* Predictability, structure preservation, latency.

*Answer sketch:* Fixed-size: simplest, fastest, ignores structure — fine for uniform text. Recursive (default): respects natural boundaries when possible — best general-purpose. Semantic (embedding-aware): finds true topic shifts — highest quality but ~10× slower (calls embeddings during chunking). Default to recursive; upgrade to semantic only when retrieval quality is failing on long heterogeneous documents.

---

**Q4. What is chunk overlap — why does it exist; what's a good default?**

*Hint:* Information bleed across boundaries.

*Answer sketch:* Overlap = the last N characters of chunk *k* are repeated as the first N of chunk *k+1*. This prevents a sentence that straddles a boundary from being split between two chunks (each of which then loses its context). Default: 10–15% of `chunk_size` — e.g., `chunk_size=800, chunk_overlap=100`. Too much overlap = duplicated content in retrieval; too little = boundary information loss.

---

**Q5. When does `MarkdownHeaderTextSplitter` beat recursive splitting?**

*Hint:* Documents with explicit, exploitable structure.

*Answer sketch:* Beats it whenever the document has stable headings that carry semantic meaning — policy docs, technical specs, knowledge-base articles, product manuals. Each chunk arrives with `Section`, `Subsection` in metadata, so the LLM sees both the chunk text *and* the heading chain. Recursive splitting may chop the heading off — the chunk content alone often doesn't tell you which section it came from.

---

**Q6. How do you decide chunk size — what's the trade-off vs retrieval quality?**

*Hint:* Granularity vs context.

*Answer sketch:* Small chunks (~200 tokens) = high precision, focused embeddings, low context per retrieved chunk. Large chunks (~1500 tokens) = blurred embeddings, fewer but richer chunks per retrieval. Empirical sweet spot for prose RAG: 500–1000 tokens with ~10% overlap. Always tune against an eval set — there's no universal answer; document length, embedding model, and downstream LLM context budget all interact.

---

**Q7. How would you chunk a CSV with thousands of rows?**

*Hint:* Don't split row content; group rows.

*Answer sketch:* Two patterns: (1) one Document per row (`CSVLoader` default) — works when each row is independently meaningful (a transaction, an FAQ pair). (2) For row-grouping, write a custom chunker: each chunk = N rows + the header row repeated. The header repetition is critical — without it, retrieved chunks lose column meaning. Pick N so the chunk fits the embedding token budget.

---

**Q8. Why do tiktoken-based splitters exist — what problem do they solve?**

*Hint:* Character ≠ token.

*Answer sketch:* Character-based splitters give *approximate* token counts. A `chunk_size=1000` characters could be 200–400 tokens depending on language and content. When the embedding model has a hard cap (e.g., 8191 tokens for `text-embedding-3-small`, or 512 for some open-source models), you need exact token counts. tiktoken splitters use the same tokenizer as the model, so `chunk_size=500` *means* exactly 500 tokens — no overflow at embed time.

